# AC4 — Neteja i anàlisi de dades d’una llibreria

## 1. Presentació i objectius

Aquest projecte analitza el fitxer `05_llibreria.csv`, que conté
informació sobre llibres registrats en diferents sucursals, amb el
gènere, el nombre d’exemplars, el nombre de pàgines i la data.

L’objectiu és obtenir dades fiables per comparar els exemplars per
gènere i sucursal, i estudiar el nombre de pàgines dels llibres
registrats.

Abans de l’anàlisi, explorarem el fitxer, identificarem les incidències
i aplicarem regles de neteja i validació. Documentarem les correccions
i els motius de rebuig, sense inventar informació per completar les dades.

### Descripció de les columnes

| Columna | Significat | Tipus esperat després de la validació |
|---|---|---|
| `titol` | Denominació del llibre o article registrat. | Text no buit. |
| `genere` | Gènere o classificació del registre. | Text no buit. |
| `sucursal` | Sucursal on s’ha registrat. | Text no buit. |
| `exemplars` | Quantitat d’exemplars del registre. | Nombre enter estrictament positiu. |
| `pagines` | Nombre de pàgines del llibre o article. | Nombre enter estrictament positiu. |
| `data` | Data associada al registre. | Data vàlida en format `AAAA-MM-DD`. |

### Relació amb l’AC3

Reutilitzarem el procediment de l’AC3: lectura del CSV, exploració,
funcions de conversió, neteja amb traçabilitat, separació de registres,
agrupacions i comprovacions de coherència.

Adaptarem els camps obligatoris i les conversions perquè tant
`exemplars` com `pagines` han de ser enters positius. També substituirem
els càlculs de preus i facturació per indicadors adequats a les dades
de la llibreria.

### Abast de les conclusions

Les conclusions descriuran els registres acceptats del fitxer.
Parlarem d’exemplars registrats, ja que les dades no especifiquen
el tipus d’operació. També valorarem com poden afectar els registres
rebutjats a la interpretació dels resultats.

### Instruccions d’execució

Cal tenir Python 3 i el fitxer `05_llibreria.csv` a la mateixa
carpeta que aquest notebook. El codi utilitza exclusivament
mòduls de la biblioteca estàndard de Python.

Les cel·les s’han d’executar de dalt a baix, perquè algunes
utilitzen funcions i variables definides en cel·les anteriors.

Per comprovar l’execució completa, cal reiniciar el nucli,
executar totes les cel·les i guardar el notebook amb els
resultats visibles.

El notebook genera tres fitxers a la carpeta de treball:
`registres_nets.csv`, `incidencies.csv` i `registres_rebutjats.csv`.
Si ja existeixen, l’exportació en substitueix el contingut.
El fitxer original `05_llibreria.csv` no es modifica.

## 2. Càrrega i exploració inicial

Llegim el CSV amb `csv.DictReader`, seguint el procediment de l’AC3.
Cada registre es desa com un diccionari, amb els noms de les columnes
com a claus.

Comprovem les dimensions, les columnes, els primers registres i els
tipus de dades llegits abans d’aplicar cap transformació.

In [1]:
import csv

nom_fitxer = "05_llibreria.csv"

# utf-8-sig permet llegir els accents i elimina la marca BOM inicial.
# newline="" deixa que el mòdul csv gestioni els salts de línia.
with open(nom_fitxer, mode="r", encoding="utf-8-sig", newline="") as fitxer:
    lector = csv.DictReader(fitxer)
    camps = lector.fieldnames
    registres = list(lector)

# Dimensions i estructura del conjunt de dades.
print("Nombre de registres:", len(registres))
print("Nombre de columnes:", len(camps) if camps else 0)
print("Columnes:", camps)

# Mostrem una petita mostra sense modificar els registres originals.
print("\nPrimers 5 registres:")
for registre in registres[:5]:
    print(registre)

# El CSV no incorpora tipus de Python: inicialment llegim els valors com a text.
if registres:
    print("\nTipus de dades del primer registre:")
    for camp, valor in registres[0].items():
        print(f"{camp}: {type(valor).__name__}")

Nombre de registres: 120
Nombre de columnes: 6
Columnes: ['titol', 'genere', 'sucursal', 'exemplars', 'pagines', 'data']

Primers 5 registres:
{'titol': 'Quadern', 'genere': 'Educació', 'sucursal': 'Vic', 'exemplars': '47', 'pagines': '18', 'data': '2026-04-03'}
{'titol': 'Còmic', 'genere': 'Coneixement', 'sucursal': 'Manresa', 'exemplars': '10', 'pagines': '79', 'data': '2025-03-20'}
{'titol': 'Còmic', 'genere': 'Còmic', 'sucursal': 'Vic', 'exemplars': '20', 'pagines': '23', 'data': '2026-05-23'}
{'titol': 'Novel·la', 'genere': 'Ficció', 'sucursal': 'Barcelona', 'exemplars': '20', 'pagines': '41', 'data': '2025-06-13'}
{'titol': 'Llibre infantil', 'genere': 'Infantil', 'sucursal': '   ', 'exemplars': '1O', 'pagines': '48', 'data': '2025-06-04'}

Tipus de dades del primer registre:
titol: str
genere: str
sucursal: str
exemplars: str
pagines: str
data: str


## 3. Detecció d’incidències

Revisem les dades originals per identificar camps buits, valors
numèrics incorrectes, dates invàlides i inconsistències de format,
abans d’aplicar les transformacions.

### 3.1. Camps buits

Comptem els registres amb almenys un camp buit o format només per
espais. També comptem els buits de cada columna per identificar
quins camps estan afectats.

Aquesta comprovació encara no determina tots els registres rebutjats:
també caldrà validar els nombres i les dates.

In [2]:
registres_incomplets = 0
buits_per_camp = {camp: 0 for camp in camps}

for registre in registres:
    te_buit = False

    for camp in camps:
        valor = registre.get(camp)

        # None indica un valor absent; strip() detecta textos amb només espais.
        if valor is None or valor.strip() == "":
            buits_per_camp[camp] += 1
            te_buit = True

    # Comptem cada registre una sola vegada, encara que tingui diversos buits.
    if te_buit:
        registres_incomplets += 1

print("Registres amb almenys un camp buit:", registres_incomplets)

print("\nCamps buits per columna:")
for camp, quantitat in buits_per_camp.items():
    print(f"{camp}: {quantitat}")

Registres amb almenys un camp buit: 15

Camps buits per columna:
titol: 4
genere: 5
sucursal: 5
exemplars: 1
pagines: 1
data: 1


<div style="color: #5DADE2; font-style: italic;">

S’han detectat 15 registres amb almenys un camp buit, que representen
el 12,5 % dels 120 registres inicials.

En total hi ha 17 camps buits: 4 a titol, 5 a genere, 5 a sucursal,
1 a exemplars, 1 a pagines i 1 a data. Algunes files tenen més
d’un camp buit.

Encara cal comprovar els valors numèrics, les dates i els formats
per determinar tots els registres que no compleixen les regles.

</div>

### 3.2. Funció de comprovació dels camps numèrics

La funció següent retorna un enter positiu si el valor és acceptable,
o `None` si no ho és.

Utilitzem `Decimal`, de la biblioteca estàndard de Python, per
interpretar els decimals amb precisió i comprovar que el valor
no tingui part fraccionària abans de convertir-lo a enter.

In [3]:
from decimal import Decimal, InvalidOperation


def converteix_enter_positiu(valor):
    """Retorna un enter positiu o None si el valor no és vàlid."""
    if valor is None:
        return None

    # Netegem els extrems i normalitzem el separador decimal.
    text = str(valor).strip().replace(",", ".")

    try:
        nombre = Decimal(text)

        # Rebutgem valors especials com NaN i Infinity.
        if not nombre.is_finite():
            return None

        # No acceptem zeros ni nombres negatius.
        if nombre <= 0:
            return None

        # Comprovem que no hi hagi part fraccionària.
        # Aquesta comprovació evita convertir 2.5 en 2.
        if nombre != nombre.to_integral_value():
            return None

        return int(nombre)

    except InvalidOperation:
        # Text buit o que no es pot interpretar com un nombre.
        return None

Fem una comprovació ràpida de la funció amb alguns valors de prova.

In [4]:
# Cada parella conté el valor original i el resultat que esperem.
casos = [
    ("47", 47),
    (" 12 ", 12),
    ("3,0", 3),
    ("3.0", 3),
    ("2.5", None),
    ("0", None),
    ("-4", None),
    ("", None),
    ("   ", None),
    ("1O", None),
    ("12 unitats", None),
    ("1,2,3", None),
    ("NaN", None),
    ("Infinity", None),
    (None, None),
]

for original, esperat in casos:
    resultat = converteix_enter_positiu(original)

    # Si algun resultat no coincideix, l'execució s'atura.
    assert resultat == esperat, (
        f"Error amb {original!r}: esperàvem {esperat}, obtenim {resultat}"
    )

    # !r mostra les cometes i permet distingir els textos buits i els espais.
    print(f"{original!r} → {resultat}")

print("\nTotes les comprovacions s’han superat.")

'47' → 47
' 12 ' → 12
'3,0' → 3
'3.0' → 3
'2.5' → None
'0' → None
'-4' → None
'' → None
'   ' → None
'1O' → None
'12 unitats' → None
'1,2,3' → None
'NaN' → None
'Infinity' → None
None → None

Totes les comprovacions s’han superat.


<div style="color: #5DADE2; font-style: italic;">

<p>Les 15 comprovacions s’han superat. La funció accepta enters
positius, elimina els espais dels extrems i converteix representacions
com «3,0» i «3.0» a l’enter 3.</p>

<p>Els valors amb part fraccionària, els zeros, els negatius, els camps
buits i els textos no interpretables retornen None. També es rebutgen
NaN i Infinity. No s’arrodoneixen quantitats ni es reconstrueixen
valors com «1O» o «12 unitats».</p>

<p>Aquestes proves comproven el comportament de la funció amb els casos
seleccionats. Encara no s’ha aplicat la validació als registres del CSV.</p>

</div>

### 3.3. Funció de comprovació de dates

Comprovem que el text tingui el format exacte `AAAA-MM-DD`
i que representi una data real del calendari.

Eliminem els espais dels extrems, però no reconstruïm dates
impossibles ni interpretem altres formats.

In [5]:
import re
from datetime import datetime


def data_valida(valor):
    """Retorna True si el valor és una data vàlida en format AAAA-MM-DD."""
    if valor is None:
        return False

    text = str(valor).strip()

    # Exigim quatre xifres per a l'any i dues per al mes i el dia.
    if re.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}", text) is None:
        return False

    try:
        # Comprovem que la data existeixi, inclosos els anys de traspàs.
        datetime.strptime(text, "%Y-%m-%d")
        return True

    except ValueError:
        return False

Ara toca comprovar-ho:

In [6]:
casos_dates = [
    ("2026-04-03", True),
    (" 2026-04-03 ", True),
    ("2024-02-29", True),   # Any de traspàs.
    ("2025-02-29", False),
    ("2025-04-31", False),
    ("2025-14-03", False),
    ("2025-00-12", False),
    ("2025-1-4", False),    # No compleix el format exacte.
    ("2025.06.10", False),
    ("31/04/2025", False),
    ("ahir", False),
    ("", False),
    (None, False),
]

for original, esperat in casos_dates:
    resultat = data_valida(original)

    assert resultat == esperat, (
        f"Error amb {original!r}: esperàvem {esperat}, obtenim {resultat}"
    )

    print(f"{original!r} → {resultat}")

print("\nTotes les comprovacions de dates s’han superat.")

'2026-04-03' → True
' 2026-04-03 ' → True
'2024-02-29' → True
'2025-02-29' → False
'2025-04-31' → False
'2025-14-03' → False
'2025-00-12' → False
'2025-1-4' → False
'2025.06.10' → False
'31/04/2025' → False
'ahir' → False
'' → False
None → False

Totes les comprovacions de dates s’han superat.


<div style="color: #5DADE2; font-style: italic;">

<p>Les 13 comprovacions s’han superat. La funció retorna True quan
el valor representa una data real en format exacte AAAA-MM-DD,
després d’eliminar els espais dels extrems. En cas contrari,
retorna False.</p>

<p>La comprovació té dues parts: revisar el format i comprovar que
la data existeixi al calendari. Per exemple, «2025-1-4» es rebutja
perquè el mes i el dia no tenen dues xifres. En canvi, «2025-04-31»
compleix el format, però es rebutja perquè abril només té 30 dies.</p>

<p>També es comproven els anys de traspàs: «2024-02-29» és una data
vàlida, mentre que «2025-02-29» no ho és. Els mesos inexistents,
els altres formats, els textos com «ahir» i els valors buits
o absents es rebutgen.</p>

<p>Aquestes proves comproven la funció amb els casos seleccionats.
Encara no s’han validat les dates del CSV. Quan l’apliquem al fitxer,
documentarem les dates rebutjades sense intentar endevinar
quin valor s’hi volia introduir.</p>

</div>

### 3.4. Detecció de valors numèrics incorrectes i dates invàlides

Apliquem les funcions de comprovació als registres originals.
Comptem els valors que no compleixen els criteris i mostrem
exemples amb la fila d’origen.

En aquest pas no modifiquem, corregim ni separem els registres.
Un mateix registre pot presentar problemes en diversos camps.

In [7]:
problemes_detectats = []

for fila, registre in enumerate(registres, start=2):
    # Comprovem els dos camps que han de representar enters positius.
    for camp in ["exemplars", "pagines"]:
        original = registre.get(camp)

        if converteix_enter_positiu(original) is None:
            problemes_detectats.append({
                "fila_csv": fila,
                "camp": camp,
                "valor_original": original
            })

    # Comprovem el format i la validesa de la data.
    if not data_valida(registre.get("data")):
        problemes_detectats.append({
            "fila_csv": fila,
            "camp": "data",
            "valor_original": registre.get("data")
        })

print("VALORS QUE NO SUPEREN LA COMPROVACIÓ")

for camp in ["exemplars", "pagines", "data"]:
    problemes_camp = [
        problema for problema in problemes_detectats
        if problema["camp"] == camp
    ]

    print(f"\n{camp}: {len(problemes_camp)} casos")

    # Mostrem dos exemples de cada camp.
    for problema in problemes_camp[:2]:
        print(
            f"  Fila {problema['fila_csv']}: "
            f"{problema['valor_original']!r}"
        )

# Comptem files diferents, sense duplicar les que tenen diversos problemes.
files_afectades = {
    problema["fila_csv"] for problema in problemes_detectats
}

print(
    "\nRegistres amb algun problema numèric o de data:",
    len(files_afectades)
)

VALORS QUE NO SUPEREN LA COMPROVACIÓ

exemplars: 11 casos
  Fila 6: '1O'
  Fila 32: '2.5'

pagines: 12 casos
  Fila 21: 'aprox. 25'
  Fila 23: '--'

data: 12 casos
  Fila 21: '31/04/2025'
  Fila 23: 'ahir'

Registres amb algun problema numèric o de data: 24


<div style="color: #5DADE2; font-style: italic;">

<p>S’han detectat 11 valors incorrectes a exemplars, 12 a pagines
i 12 a data. En total són 35 problemes repartits entre 24 registres:
algunes files tenen més d’un camp afectat.</p>

<p>Els exemples mostren problemes diferents. «1O» conté una lletra
i no es pot interpretar com un nombre; «2.5» és numèric, però no
és enter. «aprox. 25» i «--» tampoc compleixen el criteri d’enter
positiu establert per a les pàgines.</p>

<p>La data «31/04/2025» presenta un format no admès i un dia
inexistent, perquè abril té 30 dies. «ahir» no identifica
una data en el format requerit.</p>

<p>Els 24 registres afectats no s’han de sumar directament
als 15 registres amb camps buits: les dues deteccions poden
incloure les mateixes files. Encara no hem fet la classificació
final ni modificat les dades originals.</p>

</div>

### 3.5. Detecció de formats normalitzables

Identifiquem textos no buits amb espais als extrems i representacions
numèriques vàlides que necessiten normalització, com `3,0`.

Els camps formats només per espais ja s’han comptat com a buits.
Aquí només detectem els canvis possibles; encara no els apliquem
als registres.

In [8]:
formats_normalitzables = []

for fila, registre in enumerate(registres, start=2):
    # Textos obligatoris: detectem espais sobrants si hi ha contingut.
    for camp in ["titol", "genere", "sucursal"]:
        original = registre.get(camp)

        if original is not None:
            text_net = original.strip()

            if text_net != "" and original != text_net:
                formats_normalitzables.append({
                    "fila_csv": fila,
                    "camp": camp,
                    "original": original,
                    "normalitzat": text_net
                })

    # Nombres: només proposem normalitzar valors que superen la validació.
    for camp in ["exemplars", "pagines"]:
        original = registre.get(camp)
        nombre = converteix_enter_positiu(original)

        if nombre is not None and original != str(nombre):
            formats_normalitzables.append({
                "fila_csv": fila,
                "camp": camp,
                "original": original,
                "normalitzat": nombre
            })

    # Dates vàlides amb espais als extrems.
    original = registre.get("data")

    if data_valida(original) and original != original.strip():
        formats_normalitzables.append({
            "fila_csv": fila,
            "camp": "data",
            "original": original,
            "normalitzat": original.strip()
        })

print("FORMATS NORMALITZABLES\n")

for cas in formats_normalitzables:
    print(
        f"Fila {cas['fila_csv']} — {cas['camp']}: "
        f"{cas['original']!r} → {cas['normalitzat']!r}"
    )

print("\nCamps amb format normalitzable:", len(formats_normalitzables))
print(
    "Registres afectats:",
    len({cas["fila_csv"] for cas in formats_normalitzables})
)

FORMATS NORMALITZABLES

Fila 63 — exemplars: '3,0' → 3
Fila 74 — titol: ' Agenda  ' → 'Agenda'
Fila 83 — genere: ' Coneixement  ' → 'Coneixement'
Fila 91 — titol: ' Llibre de text  ' → 'Llibre de text'
Fila 92 — titol: ' Quadern  ' → 'Quadern'
Fila 96 — genere: ' Papereria  ' → 'Papereria'
Fila 113 — titol: ' Assaig  ' → 'Assaig'
Fila 114 — genere: ' Coneixement  ' → 'Coneixement'
Fila 121 — titol: ' Llibre de text  ' → 'Llibre de text'

Camps amb format normalitzable: 9
Registres afectats: 9


<div style="color: #5DADE2; font-style: italic;">

<p>S’han identificat 9 camps amb formats normalitzables,
cadascun en un registre diferent: 5 corresponen a titol,
3 a genere i 1 a exemplars.</p>

<p>En els camps textuals, la transformació proposada elimina
els espais dels extrems sense alterar el contingut. En el camp
exemplars, «3,0» es pot representar com l’enter 3 perquè
no té part fraccionària.</p>

<p>Aquests casos es diferencien dels camps buits i dels valors
no interpretables: la informació necessària hi és i es pot
normalitzar sense endevinar-la.</p>

<p>La detecció no implica que els nou registres siguin acceptables.
Cal revisar tots els seus camps abans de decidir-ho. En aquesta
fase només s’han identificat les transformacions possibles;
les dades originals no s’han modificat.</p>

<p>La numeració correspon a les línies del CSV: la fila 1
conté la capçalera i les files 2 a 121 contenen els 120 registres.</p>

</div>

## 4. Neteja i validació dels registres

Revisem tots els camps de cada registre, encara que ja hi hàgim
detectat un error, per recollir tots els motius de rebuig.

Desem els resultats en tres llistes:

- `registres_nets`: registres acceptats, amb textos nets i nombres enters.
- `rebutjades`: registres originals rebutjats, identificats per fila.
- `incidencies`: detall de les correccions i dels motius de rebuig.

La numeració comença a la fila 2 perquè la primera línia del CSV
conté les capçaleres. Les dades originals es conserven sense modificar.

### 4.1. Regles de neteja i validació

Definim les regles que han de complir els registres per poder-los incloure en l’anàlisi.

#### Camps descriptius

- `titol`, `genere` i `sucursal` són obligatoris.
- Eliminem els espais dels extrems amb `strip()`.
- Si el valor és buit després d’aquesta neteja, rebutgem el registre.
- No canviem les classificacions ni completem informació per intuïció.

#### Camps numèrics

- `exemplars` i `pagines` han de ser enters estrictament positius.
- Eliminem els espais dels extrems i normalitzem la coma decimal.
- Acceptem representacions com `3,0` perquè equivalen exactament
  a l’enter 3.
- Rebutgem valors amb part fraccionària, com `2.5`, sense arrodonir-los
  ni truncar-los.
- Rebutgem camps buits, zeros, negatius i textos no interpretables
  com a nombres.
- No substituïm lletres per xifres ni extraiem nombres de textos
  com `1O` o `12 unitats`.

#### Dates

- Eliminem els espais dels extrems.
- Exigim el format exacte `AAAA-MM-DD` i una data real del calendari.
- Rebutgem dates buides, impossibles o amb un altre format.
- No intentem endevinar ni reconstruir dates incorrectes.

#### Acceptació i traçabilitat

- Acceptem un registre només si compleix totes les regles.
- Conservem els registres originals sense modificar-los.
- Documentem les correccions i tots els motius de rebuig, identificant
  la fila del CSV, el camp afectat i el valor original.
- Una fila pot tenir diverses incidències, però es compta una sola
  vegada com a registre acceptat o rebutjat.
- Una correcció de format no garanteix l’acceptació: el registre
  pot tenir un altre camp invàlid.

In [9]:
def registra_incidencia(fila, tipus, camp, original, accio):
    """Afegeix una correcció o un motiu de rebuig al registre d'incidències."""
    incidencies.append({
        "fila_csv": fila,
        "tipus": tipus,
        "camp": camp,
        "valor_original": original,
        "accio": accio
    })

<div style="color: #5DADE2; font-style: italic;">

<p>Aquesta funció centralitza com documentem les incidències.
Rep la fila del CSV, el tipus d’incidència, el camp afectat,
el valor original i l’acció o el motiu de rebuig. Amb aquesta
informació, afegeix un diccionari a la llista incidencies.</p>

<p>La definim en una cel·la separada per distingir l’eina que
registra les incidències del procés que revisa les dades.
Així evitem repetir el mateix diccionari en cada comprovació
i mantenim una estructura uniforme.</p>

<p>Executar aquesta primera cel·la només defineix la funció:
encara no revisa registres ni genera incidències.
La cel·la següent inicialitza les llistes i crida la funció
cada vegada que detecta una correcció o un motiu de rebuig.</p>

</div>

In [10]:
# Reiniciem les llistes per no duplicar resultats en reexecutar la cel·la.
registres_nets = []
rebutjades = []
incidencies = []

obligatoris = ["titol", "genere", "sucursal"]
numerics = ["exemplars", "pagines"]

for fila, registre in enumerate(registres, start=2):
    es_valid = True

    # Creem un diccionari nou i conservem la fila d'origen.
    registre_net = {"fila_csv": fila}

    # 1. Camps descriptius obligatoris.
    for camp in obligatoris:
        original = registre.get(camp)
        text_net = "" if original is None else original.strip()

        if text_net == "":
            es_valid = False
            registra_incidencia(
                fila, "REBUTJAT", camp, original,
                "Camp obligatori buit"
            )
        else:
            registre_net[camp] = text_net

            if original != text_net:
                registra_incidencia(
                    fila, "CORRECCIO", camp, original,
                    "Espais dels extrems eliminats"
                )

    # 2. Camps numèrics: tots dos han de ser enters positius.
    for camp in numerics:
        original = registre.get(camp)
        nombre = converteix_enter_positiu(original)

        if nombre is None:
            es_valid = False
            registra_incidencia(
                fila, "REBUTJAT", camp, original,
                "Valor buit o no interpretable com a enter positiu"
            )
        else:
            registre_net[camp] = nombre

            # Detectem canvis de format, com " 12 " o "3,0".
            # La conversió habitual de "12" a int no es compta com a correcció.
            if original != str(nombre):
                registra_incidencia(
                    fila, "CORRECCIO", camp, original,
                    "Format numèric normalitzat sense canviar el valor"
                )

    # 3. Data: format exacte i validesa al calendari.
    original = registre.get("data")

    if not data_valida(original):
        es_valid = False
        registra_incidencia(
            fila, "REBUTJAT", "data", original,
            "Data buida, impossible o amb format diferent de AAAA-MM-DD"
        )
    else:
        registre_net["data"] = original.strip()

        if original != registre_net["data"]:
            registra_incidencia(
                fila, "CORRECCIO", "data", original,
                "Espais dels extrems eliminats"
            )

    # 4. Cada registre s'afegeix a una única llista final.
    if es_valid:
        registres_nets.append(registre_net)
    else:
        rebutjades.append({
            "fila_csv": fila,
            "registre_original": registre.copy()
        })

print("Registres inicials:", len(registres))
print("Registres acceptats:", len(registres_nets))
print("Registres rebutjats:", len(rebutjades))
print("Incidències documentades:", len(incidencies))

Registres inicials: 120
Registres acceptats: 90
Registres rebutjats: 30
Incidències documentades: 58


<div style="color: #5DADE2; font-style: italic;">

<p>Dels 120 registres inicials, se n’han acceptat 90, el 75 %,
i se n’han rebutjat 30, el 25 %. Els acceptats compleixen totes
les regles després de les normalitzacions permeses.
Els rebutjats incompleixen almenys una regla.</p>

<p>S’han documentat 58 incidències. Aquesta xifra inclou tant
correccions de format com motius de rebuig; per tant, no equival
al nombre de registres rebutjats. Una mateixa fila pot tenir
diverses incidències, i una correcció també pot aparèixer
en un registre acceptat.</p>

<p>La suma dels acceptats i els rebutjats coincideix amb els
120 registres inicials. L’anàlisi posterior utilitzarà els
90 registres acceptats. Cal tenir present que l’exclusió
del 25 % de les files pot afectar les comparacions.</p>

<p>El recompte global encara no indica quins problemes són
els més freqüents. A continuació revisarem el detall de les
incidències i els motius pels quals s’han rebutjat registres.</p>

</div>

### 4.2. Mostra dels registres nets

Mostrem els primers cinc registres acceptats i comprovem els tipus
del primer. Els camps exemplars i pagines han de contenir enters;
els camps descriptius i la data es conserven com a text.
El camp fila_csv identifica la línia del fitxer original.

In [11]:
print("PRIMERS CINC REGISTRES ACCEPTATS\n")

for registre in registres_nets[:5]:
    print(registre)

if registres_nets:
    print("\nTIPUS DE DADES DEL PRIMER REGISTRE NET\n")

    for camp, valor in registres_nets[0].items():
        print(f"{camp}: {type(valor).__name__}")

PRIMERS CINC REGISTRES ACCEPTATS

{'fila_csv': 2, 'titol': 'Quadern', 'genere': 'Educació', 'sucursal': 'Vic', 'exemplars': 47, 'pagines': 18, 'data': '2026-04-03'}
{'fila_csv': 3, 'titol': 'Còmic', 'genere': 'Coneixement', 'sucursal': 'Manresa', 'exemplars': 10, 'pagines': 79, 'data': '2025-03-20'}
{'fila_csv': 4, 'titol': 'Còmic', 'genere': 'Còmic', 'sucursal': 'Vic', 'exemplars': 20, 'pagines': 23, 'data': '2026-05-23'}
{'fila_csv': 5, 'titol': 'Novel·la', 'genere': 'Ficció', 'sucursal': 'Barcelona', 'exemplars': 20, 'pagines': 41, 'data': '2025-06-13'}
{'fila_csv': 7, 'titol': 'Llibre de text', 'genere': 'Papereria', 'sucursal': 'Barcelona', 'exemplars': 8, 'pagines': 48, 'data': '2025-02-23'}

TIPUS DE DADES DEL PRIMER REGISTRE NET

fila_csv: int
titol: str
genere: str
sucursal: str
exemplars: int
pagines: int
data: str


<div style="color: #5DADE2; font-style: italic;">

<p>La mostra presenta els primers cinc registres acceptats.
Exemplars i pagines ja són de tipus int, mentre que els camps
descriptius es conserven com a text.</p>

<p>La data també es conserva com a text, després de comprovar
que representa una data real en format AAAA-MM-DD. Validar-la
no implica convertir-la a un objecte de data de Python.</p>

<p>El camp fila_csv manté la referència al fitxer original.
La numeració passa de la fila 5 a la 7 perquè la fila 6
ha estat rebutjada; els registres acceptats no es renumeren.</p>

<p>Aquesta mostra permet observar l’estructura de les dades
netes. Les comprovacions de coherència del punt 5.3 verifiquen
les condicions sobre tots els registres acceptats.</p>

</div>

## 5. Gestió dels registres incorrectes

Revisem els registres rebutjats i els motius que impedeixen
acceptar-los. També distingim els errors de les correccions
de format i comprovem la coherència de la classificació.

### 5.1. Distribució de les incidències

Separem les correccions de format dels motius de rebuig i comptem
quantes incidències afecten cada camp.

Aquests recomptes corresponen a incidències, no a files diferents:
un registre pot presentar problemes en diversos camps.

In [12]:
# Separem les incidències segons el tipus.
motius_rebuig = [
    incidencia for incidencia in incidencies
    if incidencia["tipus"] == "REBUTJAT"
]

correccions = [
    incidencia for incidencia in incidencies
    if incidencia["tipus"] == "CORRECCIO"
]

print("Motius de rebuig documentats:", len(motius_rebuig))
print("Correccions de format documentades:", len(correccions))

# Comptem les incidències de cada tipus per columna.
rebutjos_per_camp = {camp: 0 for camp in camps}
correccions_per_camp = {camp: 0 for camp in camps}

for incidencia in motius_rebuig:
    rebutjos_per_camp[incidencia["camp"]] += 1

for incidencia in correccions:
    correccions_per_camp[incidencia["camp"]] += 1

print("\nIncidències per camp:")
print(f"{'Camp':<15} {'Motius de rebuig':>18} {'Correccions':>13}")

for camp in camps:
    print(
        f"{camp:<15} "
        f"{rebutjos_per_camp[camp]:>18} "
        f"{correccions_per_camp[camp]:>13}"
    )

# Comprovem que la separació inclogui totes les incidències.
assert len(motius_rebuig) + len(correccions) == len(incidencies), (
    "La suma dels motius de rebuig i les correccions no quadra."
)

Motius de rebuig documentats: 49
Correccions de format documentades: 9

Incidències per camp:
Camp              Motius de rebuig   Correccions
titol                            4             5
genere                           5             3
sucursal                         5             0
exemplars                       11             1
pagines                         12             0
data                            12             0


<div style="color: #5DADE2; font-style: italic;">

<p>Les 58 incidències es divideixen en 49 motius de rebuig i
9 correccions de format. Els 49 motius corresponen a 30 registres
rebutjats, perquè una mateixa fila pot incomplir diverses regles.</p>

<p>Els camps amb més motius de rebuig són pagines i data,
amb 12 cadascun, seguits d’exemplars, amb 11. Els camps descriptius
acumulen 14 motius de rebuig per informació obligatòria buida:
4 a titol, 5 a genere i 5 a sucursal.</p>

<p>Les correccions es concentren en titol, amb 5 casos, i genere,
amb 3, on s’han eliminat espais dels extrems. També s’ha normalitzat
un valor d’exemplars. Aquestes correccions conserven la informació
original i no impliquen necessàriament que tota la fila sigui vàlida.</p>

<p>Els resultats indiquen que la revisió de nombres i dates és
especialment rellevant en aquest fitxer. Per millorar les dades
en origen, convindria validar els enters positius i les dates
en introduir-los, i impedir que els camps obligatoris quedin buits.</p>

<p>La taula identifica els camps afectats, però encara no diferencia
els tipus concrets d’error dins de cada camp. Revisarem exemples
per explicar què ha provocat els rebutjos i les correccions.</p>

</div>

### 5.2. Exemples de correccions i registres rebutjats

Mostrem una correcció de text, una normalització numèrica i els
primers tres registres rebutjats amb tots els seus motius.

La fila del CSV permet relacionar cada incidència amb el registre
original. Els exemples il·lustren les decisions; no representen
totes les incidències del fitxer.

In [13]:
# Busquem un exemple de cada tipus de correcció.
exemple_text = next(
    (inc for inc in correccions if inc["camp"] in obligatoris),
    None
)

exemple_numeric = next(
    (inc for inc in correccions if inc["camp"] in numerics),
    None
)

print("EXEMPLES DE CORRECCIONS")

for exemple in [exemple_text, exemple_numeric]:
    if exemple is not None:
        fila = exemple["fila_csv"]
        camp = exemple["camp"]

        # Recuperem el valor normalitzat, encara que la fila s'hagi rebutjat.
        original = exemple["valor_original"]
        if camp in numerics:
            valor_net = converteix_enter_positiu(original)
        else:
            valor_net = original.strip()

        print(f"\nFila {fila} — camp: {camp}")
        print(f"Valor original: {original!r}")
        print(f"Valor normalitzat: {valor_net!r}")
        print("Acció:", exemple["accio"])

        # Una correcció no implica que la resta de la fila sigui vàlida.
        fila_acceptada = any(
            reg["fila_csv"] == fila for reg in registres_nets
        )
        print("Estat final:", "Acceptat" if fila_acceptada else "Rebutjat")

print("\nEXEMPLES DE REGISTRES REBUTJATS")

for rebutjada in rebutjades[:3]:
    fila = rebutjada["fila_csv"]

    print(f"\nFila {fila}")
    print("Registre original:", rebutjada["registre_original"])
    print("Motius de rebuig:")

    # Mostrem tots els motius associats a aquesta fila.
    for incidencia in motius_rebuig:
        if incidencia["fila_csv"] == fila:
            print(
                f"  - {incidencia['camp']}: "
                f"{incidencia['valor_original']!r} — "
                f"{incidencia['accio']}"
            )

EXEMPLES DE CORRECCIONS

Fila 74 — camp: titol
Valor original: ' Agenda  '
Valor normalitzat: 'Agenda'
Acció: Espais dels extrems eliminats
Estat final: Acceptat

Fila 63 — camp: exemplars
Valor original: '3,0'
Valor normalitzat: 3
Acció: Format numèric normalitzat sense canviar el valor
Estat final: Rebutjat

EXEMPLES DE REGISTRES REBUTJATS

Fila 6
Registre original: {'titol': 'Llibre infantil', 'genere': 'Infantil', 'sucursal': '   ', 'exemplars': '1O', 'pagines': '48', 'data': '2025-06-04'}
Motius de rebuig:
  - sucursal: '   ' — Camp obligatori buit
  - exemplars: '1O' — Valor buit o no interpretable com a enter positiu

Fila 21
Registre original: {'titol': 'Novel·la', 'genere': 'Educació', 'sucursal': 'Barcelona', 'exemplars': '14', 'pagines': 'aprox. 25', 'data': '31/04/2025'}
Motius de rebuig:
  - pagines: 'aprox. 25' — Valor buit o no interpretable com a enter positiu
  - data: '31/04/2025' — Data buida, impossible o amb format diferent de AAAA-MM-DD

Fila 22
Registre original:

<div style="color: #5DADE2; font-style: italic;">

<p>A la fila 74, el títol « Agenda  » es normalitza a «Agenda»
eliminant els espais dels extrems. El registre s’accepta perquè
també compleix la resta de les regles.</p>

<p>A la fila 63, «3,0» es converteix a l’enter 3 sense alterar
la quantitat. Tot i això, el registre es rebutja perquè pagines
conté «-12.5», un valor negatiu i amb part fraccionària.
Aquest exemple mostra que corregir un camp no implica
acceptar el registre complet.</p>

<p>La fila 6 presenta dos motius de rebuig: la sucursal només
conté espais i exemplars conté «1O», amb la lletra O.
No substituïm aquesta lletra per un zero perquè implicaria
suposar quin valor es volia introduir.</p>

<p>La fila 21 també acumula dos problemes. «aprox. 25» no és
una quantitat entera exacta interpretable segons les regles.
La data «31/04/2025» té un format no admès i, a més, és impossible,
perquè abril només té 30 dies.</p>

<p>La fila 22 es rebutja perquè el títol és buit, encara que
la resta dels camps superin les comprovacions. El títol és
obligatori i no disposem d’informació per completar-lo.</p>

<p>Aquests exemples mostren que les correccions permeses
normalitzen el format sense inventar informació. Els registres
només s’accepten quan tots els camps compleixen les regles,
i els motius de rebuig queden vinculats a la fila original.</p>

</div>

### 5.3. Comprovacions de coherència

Utilitzem assercions per comprovar que:

- Tots els registres originals s’han classificat una sola vegada.
- Cap fila apareix alhora com a acceptada i rebutjada.
- Els camps obligatoris dels registres acceptats estan nets i no són buits.
- Els dos camps numèrics contenen enters estrictament positius.
- Les dates acceptades compleixen les regles.
- Tots els registres rebutjats tenen almenys un motiu documentat.

Si una condició no es compleix, `assert` atura l’execució i mostra
el missatge corresponent.

In [14]:
# 1. Comprovem la classificació de les files originals.
files_netes = [reg["fila_csv"] for reg in registres_nets]
files_rebutjades = [reg["fila_csv"] for reg in rebutjades]

# set() conserva els identificadors únics.
assert len(set(files_netes)) == len(files_netes), (
    "Hi ha files duplicades entre els registres acceptats."
)
assert len(set(files_rebutjades)) == len(files_rebutjades), (
    "Hi ha files duplicades entre els registres rebutjats."
)

assert set(files_netes).isdisjoint(files_rebutjades), (
    "Una fila apareix com a acceptada i rebutjada alhora."
)

files_esperades = set(range(2, len(registres) + 2))
assert set(files_netes) | set(files_rebutjades) == files_esperades, (
    "La classificació no coincideix amb les files originals."
)
print("Classificació completa, sense omissions ni duplicacions.")

# 2. Comprovem els textos obligatoris.
assert all(
    isinstance(reg[camp], str)
    and reg[camp] != ""
    and reg[camp] == reg[camp].strip()
    for reg in registres_nets
    for camp in obligatoris
), "Hi ha camps obligatoris buits o sense netejar."
print("Camps obligatoris nets i no buits.")

# 3. Comprovem el tipus i el valor dels camps numèrics.
assert all(
    type(reg[camp]) is int and reg[camp] > 0
    for reg in registres_nets
    for camp in numerics
), "Hi ha valors numèrics que no són enters positius."
print("Exemplars i pàgines són enters positius.")

# 4. Comprovem les dates acceptades.
assert all(
    data_valida(reg["data"])
    for reg in registres_nets
), "Hi ha dates acceptades que no compleixen les regles."
print("Dates acceptades vàlides.")

# 5. Comprovem que els rebutjos tinguin motius documentats.
files_amb_motiu = {inc["fila_csv"] for inc in motius_rebuig}

assert files_amb_motiu == set(files_rebutjades), (
    "Els motius de rebuig no coincideixen amb les files rebutjades."
)
print("Tots els registres rebutjats tenen motius documentats.")

print("\nTotes les comprovacions de coherència s’han superat.")

Classificació completa, sense omissions ni duplicacions.
Camps obligatoris nets i no buits.
Exemplars i pàgines són enters positius.
Dates acceptades vàlides.
Tots els registres rebutjats tenen motius documentats.

Totes les comprovacions de coherència s’han superat.


<div style="color: #5DADE2; font-style: italic;">

<p>Les comprovacions confirmen que tots els registres originals
s’han classificat una sola vegada, sense omissions ni duplicacions.
Cap fila apareix alhora com a acceptada i rebutjada.</p>

<p>En els registres acceptats, els camps obligatoris no són buits
i no tenen espais als extrems. Exemplars i pagines contenen enters
estrictament positius, i les dates superen la validació establerta.</p>

<p>També s’ha comprovat que totes les files rebutjades tenen
almenys un motiu documentat i que no hi ha motius de rebuig
associats a files acceptades.</p>

<p>Aquests controls verifiquen la coherència del procés amb les
regles definides. No demostren que les dades siguin certes en
origen, però ens permeten continuar l’anàlisi amb els 90 registres
que compleixen els criteris establerts.</p>

</div>

### 5.4. Exportació dels resultats

Generem tres CSV per consultar les dades resultants i el detall
complet de les incidències:

- `registres_nets.csv`: registres acceptats i normalitzats.
- `incidencies.csv`: correccions i motius de rebuig.
- `registres_rebutjats.csv`: registres originals rebutjats.

El camp `fila_csv` permet relacionar els fitxers amb les files
del CSV original. Les exportacions no modifiquen el fitxer d’entrada.
En tornar a executar aquesta cel·la, se substitueixen les
exportacions anteriors amb aquests mateixos noms.

In [15]:
# 1. Exportem els registres acceptats.
# Definim les columnes explícitament per mantenir un ordre estable
# i poder generar la capçalera encara que la llista sigui buida.
camps_registres = [
    "fila_csv", "titol", "genere", "sucursal",
    "exemplars", "pagines", "data"
]

with open(
    "registres_nets.csv", mode="w", encoding="utf-8-sig", newline=""
) as fitxer:
    escriptor = csv.DictWriter(fitxer, fieldnames=camps_registres)
    escriptor.writeheader()
    escriptor.writerows(registres_nets)

# 2. Exportem totes les correccions i els motius de rebuig.
camps_incidencies = [
    "fila_csv", "tipus", "camp", "valor_original", "accio"
]

with open(
    "incidencies.csv", mode="w", encoding="utf-8-sig", newline=""
) as fitxer:
    escriptor = csv.DictWriter(fitxer, fieldnames=camps_incidencies)
    escriptor.writeheader()
    escriptor.writerows(incidencies)

# 3. Preparem les files rebutjades per exportar-les.
# Ajuntem la fila d'origen amb els camps originals, sense netejar-los.
files_rebutjades_exportacio = []

for rebutjada in rebutjades:
    fila_exportacio = {
        "fila_csv": rebutjada["fila_csv"],
        **rebutjada["registre_original"]
    }
    files_rebutjades_exportacio.append(fila_exportacio)

with open(
    "registres_rebutjats.csv", mode="w", encoding="utf-8-sig", newline=""
) as fitxer:
    escriptor = csv.DictWriter(fitxer, fieldnames=camps_registres)
    escriptor.writeheader()
    escriptor.writerows(files_rebutjades_exportacio)

print(f"registres_nets.csv: {len(registres_nets)} registres exportats.")
print(f"incidencies.csv: {len(incidencies)} incidències exportades.")
print(
    "registres_rebutjats.csv: "
    f"{len(files_rebutjades_exportacio)} registres exportats."
)

registres_nets.csv: 90 registres exportats.
incidencies.csv: 58 incidències exportades.
registres_rebutjats.csv: 30 registres exportats.


<div style="color: #5DADE2; font-style: italic;">

<p>S’han exportat 90 registres acceptats, 58 incidències
i 30 registres rebutjats. Els fitxers permeten consultar
el detall complet fora del notebook i relacionar-lo amb
el CSV original mitjançant fila_csv.</p>

<p>Les incidències inclouen correccions i motius de rebuig.
Per consultar per què es rebutja una fila, cal buscar-ne
el número a incidencies.csv. A registres_rebutjats.csv
es conserva el contingut original d’aquella fila.</p>

<p>A continuació tornem a llegir els fitxers per comprovar
que les columnes i el contingut coincideixin amb les dades
que volíem exportar.</p>

</div>

In [16]:
exportacions = [
    ("registres_nets.csv", camps_registres, registres_nets),
    ("incidencies.csv", camps_incidencies, incidencies),
    (
        "registres_rebutjats.csv",
        camps_registres,
        files_rebutjades_exportacio
    )
]

for nom, columnes, dades_originals in exportacions:
    with open(nom, mode="r", encoding="utf-8-sig", newline="") as fitxer:
        lector = csv.DictReader(fitxer)
        columnes_llegides = lector.fieldnames
        files_llegides = list(lector)

    # En un CSV els valors es recuperen com a text.
    # DictWriter escriu None com un camp buit.
    files_esperades = [
        {
            camp: "" if registre[camp] is None else str(registre[camp])
            for camp in columnes
        }
        for registre in dades_originals
    ]

    assert columnes_llegides == columnes, (
        f"Les columnes de {nom} no coincideixen."
    )
    assert files_llegides == files_esperades, (
        f"El contingut de {nom} no coincideix amb les dades exportades."
    )

    print(f"{nom}: columnes i contingut verificats ({len(files_llegides)} files).")

print("\nTotes les exportacions s’han verificat correctament.")

registres_nets.csv: columnes i contingut verificats (90 files).
incidencies.csv: columnes i contingut verificats (58 files).
registres_rebutjats.csv: columnes i contingut verificats (30 files).

Totes les exportacions s’han verificat correctament.


<div style="color: #5DADE2; font-style: italic;">

<p>La lectura dels tres CSV confirma que les columnes i tots
els valors coincideixen amb les dades preparades per exportar:
90 registres nets, 58 incidències i 30 registres rebutjats.</p>

<p>La comprovació compara el contingut complet, no només el nombre
de files. També verifica que els valors originals dels registres
rebutjats es conservin, inclosos els espais.</p>

<p>En tornar a llegir un CSV, els valors es recuperen com a text.
Això és propi del format i no modifica els tipus de les llistes
del notebook, que continuen disponibles per a l’anàlisi.</p>

</div>

## 6. Anàlisi de les dades netes

L’anàlisi utilitza exclusivament els registres acceptats.

### 6.1. Distribució dels exemplars per gènere

Sumem els exemplars de cada gènere i calculem quin percentatge
representen sobre el total.

També mostrem el nombre de registres de cada grup per distingir
entre quantitat de files i quantitat d’exemplars.

In [17]:
total_exemplars = sum(reg["exemplars"] for reg in registres_nets)

# Cada gènere tindrà dos recomptes: exemplars i registres.
resum_genere = {}

for registre in registres_nets:
    genere = registre["genere"]

    if genere not in resum_genere:
        resum_genere[genere] = {
            "exemplars": 0,
            "registres": 0
        }

    resum_genere[genere]["exemplars"] += registre["exemplars"]
    resum_genere[genere]["registres"] += 1

# Ordenem els gèneres de més a menys exemplars.
generes_ordenats = sorted(
    resum_genere.items(),
    key=lambda element: element[1]["exemplars"],
    reverse=True
)

print("Total d’exemplars registrats:", total_exemplars)
print("Total de registres analitzats:", len(registres_nets))

print("\nDistribució per gènere:")
print(
    f"{'Gènere':<18} {'Registres':>10} "
    f"{'Exemplars':>10} {'% exemplars':>13}"
)

for genere, dades in generes_ordenats:
    percentatge = (
        dades["exemplars"] / total_exemplars * 100
        if total_exemplars > 0 else 0
    )

    print(
        f"{genere:<18} {dades['registres']:>10} "
        f"{dades['exemplars']:>10} {percentatge:>12.2f}%"
    )

# Comprovem que les agrupacions incloguin totes les dades analitzades.
assert sum(d["exemplars"] for d in resum_genere.values()) == total_exemplars, (
    "La suma d’exemplars per gènere no coincideix amb el total."
)
assert sum(d["registres"] for d in resum_genere.values()) == len(registres_nets), (
    "La suma de registres per gènere no coincideix amb el total."
)

print("\nEls totals per gènere coincideixen amb els totals globals.")

Total d’exemplars registrats: 3377
Total de registres analitzats: 90

Distribució per gènere:
Gènere              Registres  Exemplars   % exemplars
Coneixement                24        831        24.61%
Educació                   15        649        19.22%
Infantil                   13        586        17.35%
Còmic                      14        476        14.10%
Papereria                  12        475        14.07%
Ficció                     12        360        10.66%

Els totals per gènere coincideixen amb els totals globals.


<div style="color: #5DADE2; font-style: italic;">

<p>Els 90 registres acceptats acumulen 3.377 exemplars.
Coneixement és el gènere amb més exemplars registrats:
831, que representen el 24,61 % del total. El segueixen
Educació, amb 649 exemplars (19,22 %), i Infantil,
amb 586 (17,35 %).</p>

<p>Ficció presenta la quantitat més baixa, amb 360 exemplars
(10,66 %). Còmic i Papereria tenen totals gairebé iguals:
476 i 475 exemplars, respectivament, tot i que corresponen
a 14 i 12 registres.</p>

<p>El nombre de registres i la quantitat d’exemplars mesuren
coses diferents. Cada registre pot contenir diversos exemplars;
per això, tenir més files no implica necessàriament acumular-ne
més. Per exemple, Educació té 15 registres i 649 exemplars,
mentre que Còmic té 14 registres i 476 exemplars.</p>

<p>Les comprovacions confirmen que les sumes per gènere coincideixen
amb els totals globals, tant d’exemplars com de registres.</p>

<p>Aquests resultats descriuen la distribució dels exemplars
en les dades acceptades. No permeten afirmar quin gènere es ven
més ni quin prefereixen els clients, perquè el fitxer no identifica
el tipus d’operació. Tampoc inclouen els 30 registres rebutjats.</p>

</div>

### 6.2. Distribució dels exemplars per sucursal

Agrupem els registres acceptats per sucursal i calculem el nombre
de registres, el total d’exemplars i el percentatge sobre el total.

Aquesta comparació descriu el volum registrat a cada sucursal
dins del conjunt de dades analitzat.

In [18]:
resum_sucursal = {}

for registre in registres_nets:
    sucursal = registre["sucursal"]

    # Inicialitzem els comptadors quan apareix una sucursal nova.
    if sucursal not in resum_sucursal:
        resum_sucursal[sucursal] = {
            "exemplars": 0,
            "registres": 0
        }

    resum_sucursal[sucursal]["exemplars"] += registre["exemplars"]
    resum_sucursal[sucursal]["registres"] += 1

# Ordenem les sucursals de més a menys exemplars.
sucursals_ordenades = sorted(
    resum_sucursal.items(),
    key=lambda element: element[1]["exemplars"],
    reverse=True
)

print("Total d’exemplars registrats:", total_exemplars)
print("Total de registres analitzats:", len(registres_nets))

print("\nDistribució per sucursal:")
print(
    f"{'Sucursal':<18} {'Registres':>10} "
    f"{'Exemplars':>10} {'% exemplars':>13}"
)

for sucursal, dades in sucursals_ordenades:
    percentatge = (
        dades["exemplars"] / total_exemplars * 100
        if total_exemplars > 0 else 0
    )

    print(
        f"{sucursal:<18} {dades['registres']:>10} "
        f"{dades['exemplars']:>10} {percentatge:>12.2f}%"
    )

# Verifiquem que l’agrupació inclogui tots els exemplars i registres.
assert sum(d["exemplars"] for d in resum_sucursal.values()) == total_exemplars, (
    "La suma d’exemplars per sucursal no coincideix amb el total."
)
assert sum(d["registres"] for d in resum_sucursal.values()) == len(registres_nets), (
    "La suma de registres per sucursal no coincideix amb el total."
)

print("\nEls totals per sucursal coincideixen amb els totals globals.")

Total d’exemplars registrats: 3377
Total de registres analitzats: 90

Distribució per sucursal:
Sucursal            Registres  Exemplars   % exemplars
Vic                        21       1032        30.56%
Manresa                    22        849        25.14%
Barcelona                  18        639        18.92%
Tarragona                  13        510        15.10%
Girona                     16        347        10.28%

Els totals per sucursal coincideixen amb els totals globals.


<div style="color: #5DADE2; font-style: italic;">

<p>Vic concentra més exemplars registrats: 1.032, que representen
el 30,56 % del total. La segueix Manresa amb 849 exemplars
(25,14 %). Entre totes dues acumulen el 55,70 % dels exemplars
analitzats.</p>

<p>Barcelona registra 639 exemplars (18,92 %), Tarragona en
registra 510 (15,10 %) i Girona, 347 (10,28 %).</p>

<p>Manresa presenta més registres acceptats que Vic, 22 davant
de 21, però acumula menys exemplars. També Girona té més
registres que Tarragona, 16 davant de 13, però menys exemplars.
Per tant, el nombre de files no determina per si sol el volum
d’exemplars de cada sucursal.</p>

<p>Les sumes per sucursal coincideixen amb els totals globals:
90 registres i 3.377 exemplars.</p>

<p>Les diferències descriuen el contingut dels registres acceptats.
No demostren que una sucursal funcioni millor que una altra:
desconeixem el tipus d’operació i el context de cada establiment.
A més, els registres rebutjats podrien afectar la distribució
observada.</p>

</div>

### 6.3. Comparació de gèneres entre sucursals

Creuem sucursal i gènere per mostrar quants exemplars registra
cada combinació.

Després identifiquem el gènere amb més exemplars de cada sucursal
i el percentatge que representa dins d’aquella sucursal.
Si hi ha empat, mostrem tots els gèneres que comparteixen el màxim.

In [19]:
# Ordenem les etiquetes per presentar una taula fàcil de consultar.
generes = sorted(resum_genere)
sucursals = sorted(resum_sucursal)

# Preparem totes les combinacions amb zero exemplars.
exemplars_sucursal_genere = {
    sucursal: {genere: 0 for genere in generes}
    for sucursal in sucursals
}

# Acumulem els exemplars en la combinació corresponent.
for registre in registres_nets:
    sucursal = registre["sucursal"]
    genere = registre["genere"]

    exemplars_sucursal_genere[sucursal][genere] += registre["exemplars"]

# Mostrem la taula creuada.
print("EXEMPLARS PER SUCURSAL I GÈNERE\n")

capcalera = f"{'Sucursal':<14}"
for genere in generes:
    capcalera += f"{genere:>14}"
capcalera += f"{'Total':>10}"
print(capcalera)

for sucursal in sucursals:
    dades = exemplars_sucursal_genere[sucursal]
    linia = f"{sucursal:<14}"

    for genere in generes:
        linia += f"{dades[genere]:>14}"

    linia += f"{sum(dades.values()):>10}"
    print(linia)

# Comprovem les sumes de les files i de les columnes.
for sucursal in sucursals:
    assert (
        sum(exemplars_sucursal_genere[sucursal].values())
        == resum_sucursal[sucursal]["exemplars"]
    ), f"El total de {sucursal} no coincideix."

for genere in generes:
    assert (
        sum(exemplars_sucursal_genere[s][genere] for s in sucursals)
        == resum_genere[genere]["exemplars"]
    ), f"El total de {genere} no coincideix."

print("\nEls totals coincideixen amb els resums anteriors.")

# Identifiquem els màxims de cada sucursal, inclosos els possibles empats.
print("\nGÈNERES AMB MÉS EXEMPLARS DINS DE CADA SUCURSAL\n")

for sucursal in sucursals:
    dades = exemplars_sucursal_genere[sucursal]
    maxim = max(dades.values())

    destacats = [
        genere for genere, exemplars in dades.items()
        if exemplars == maxim
    ]

    total_sucursal = resum_sucursal[sucursal]["exemplars"]
    percentatge = maxim / total_sucursal * 100

    print(
        f"{sucursal}: {', '.join(destacats)} — "
        f"{maxim} exemplars ({percentatge:.2f}% de la sucursal)"
    )

EXEMPLARS PER SUCURSAL I GÈNERE

Sucursal         Coneixement         Còmic      Educació        Ficció      Infantil     Papereria     Total
Barcelona                159           143            53            85           112            87       639
Girona                   107           114            23            16            87             0       347
Manresa                  252           157           100            99           136           105       849
Tarragona                120            17            63            83           118           109       510
Vic                      193            45           410            77           133           174      1032

Els totals coincideixen amb els resums anteriors.

GÈNERES AMB MÉS EXEMPLARS DINS DE CADA SUCURSAL

Barcelona: Coneixement — 159 exemplars (24.88% de la sucursal)
Girona: Còmic — 114 exemplars (32.85% de la sucursal)
Manresa: Coneixement — 252 exemplars (29.68% de la sucursal)
Tarragona: Coneixement — 120 exemp

<div style="color: #5DADE2; font-style: italic;">

<p>Coneixement acumula més exemplars a Barcelona, amb 159
(24,88 % de la sucursal), a Manresa, amb 252 (29,68 %),
i a Tarragona, amb 120 (23,53 %). En canvi, a Girona destaca
Còmic, amb 114 exemplars (32,85 %), i a Vic, Educació,
amb 410 (39,73 %).</p>

<p>Per tant, el predomini global de Coneixement no es repeteix
a totes les sucursals. El creuament permet observar diferències
que quedaven amagades en els totals per gènere.</p>

<p>Vic concentra 410 dels 649 exemplars d’Educació del conjunt
analitzat, aproximadament el 63,17 %. Aquesta sucursal té,
doncs, un pes important en el total d’aquest gènere.</p>

<p>A Tarragona, Coneixement i Infantil presenten quantitats
molt properes: 120 i 118 exemplars. El primer lloc de Coneixement
no implica una diferència àmplia respecte d’Infantil.</p>

<p>Girona no té exemplars de Papereria entre els registres
acceptats. Aquest zero descriu les dades analitzades; no demostra
que la sucursal no disposi d’articles d’aquest gènere.</p>

<p>Els percentatges del resum es calculen sobre el total de cada
sucursal. Les comprovacions confirmen que les sumes de les files
i les columnes coincideixen amb els resums anteriors. Les diferències
observades no permeten atribuir preferències als clients ni explicar
les causes sense informació addicional.</p>

</div>

### 6.4. Distribució del nombre de pàgines

Calculem el mínim, el màxim, la mitjana i la mediana de pàgines
dels registres acceptats.

La mitjana resumeix el conjunt de valors i la mediana indica
el valor central quan els ordenem. Comparar-les ajuda a interpretar
la distribució.

Cada registre té el mateix pes en aquests càlculs, independentment
del nombre d’exemplars que contingui. No estem identificant llibres
únics: una mateixa denominació pot aparèixer en diversos registres.

Per complementar aquestes estadístiques, agrupem els registres en quatre intervals de 20 pàgines que cobreixen el rang observat, d’1 a 80.

In [20]:
from statistics import mean, median

# Prenem un valor de pàgines per cada registre acceptat.
pagines = [registre["pagines"] for registre in registres_nets]

if pagines:
    minim_pagines = min(pagines)
    maxim_pagines = max(pagines)
    mitjana_pagines = mean(pagines)
    mediana_pagines = median(pagines)

    print("Registres analitzats:", len(pagines))
    print("Mínim de pàgines:", minim_pagines)
    print("Màxim de pàgines:", maxim_pagines)
    print(f"Mitjana de pàgines per registre: {mitjana_pagines:.2f}")
    print(f"Mediana de pàgines: {mediana_pagines:.2f}")

    # Dividim el rang observat en quatre intervals consecutius.
    # Són intervals descriptius, no categories oficials de llibres.
    intervals_pagines = {
        "1–20": 0,
        "21–40": 0,
        "41–60": 0,
        "61–80": 0
    }

    for valor in pagines:
        if valor <= 20:
            intervals_pagines["1–20"] += 1
        elif valor <= 40:
            intervals_pagines["21–40"] += 1
        elif valor <= 60:
            intervals_pagines["41–60"] += 1
        elif valor <= 80:
            intervals_pagines["61–80"] += 1

    print("\nDistribució per intervals de pàgines:")
    print(f"{'Pàgines':<12} {'Registres':>10} {'% registres':>13}")

    for interval, quantitat in intervals_pagines.items():
        percentatge = quantitat / len(pagines) * 100
        print(f"{interval:<12} {quantitat:>10} {percentatge:>12.2f}%")

    # Detectem si algun valor ha quedat fora dels intervals.
    assert sum(intervals_pagines.values()) == len(pagines), (
        "Els intervals no inclouen tots els registres."
    )

    print("\nTots els registres estan inclosos en els intervals.")

else:
    print("No hi ha registres acceptats per analitzar les pàgines.")

Registres analitzats: 90
Mínim de pàgines: 1
Màxim de pàgines: 80
Mitjana de pàgines per registre: 37.12
Mediana de pàgines: 38.00

Distribució per intervals de pàgines:
Pàgines       Registres   % registres
1–20                 30        33.33%
21–40                19        21.11%
41–60                23        25.56%
61–80                18        20.00%

Tots els registres estan inclosos en els intervals.


<div style="color: #5DADE2; font-style: italic;">

<p>Els 90 registres acceptats presenten entre 1 i 80 pàgines.
La mitjana és de 37,12 pàgines per registre i la mediana és de 38.
Són valors propers, tot i que aquesta proximitat no demostra,
per si sola, que la distribució sigui simètrica.</p>

<p>L’interval d’1 a 20 pàgines és el més freqüent, amb 30 registres,
el 33,33 % del total. El segueix l’interval de 41 a 60 pàgines,
amb 23 registres (25,56 %). Els intervals de 21 a 40 i de 61 a 80
contenen 19 i 18 registres, respectivament.</p>

<p>Els quatre intervals cobreixen tots els registres analitzats.
S’han utilitzat intervals de la mateixa amplada per facilitar
la comparació; no representen categories oficials de llibres.</p>

<p>Cada registre té el mateix pes en aquests càlculs. Per tant,
la mitjana descriu les pàgines per registre, sense ponderar-les
pel nombre d’exemplars ni considerar que cada fila sigui
un llibre únic.</p>

<p>Valors com una sola pàgina poden resultar poc habituals,
però compleixen la regla d’enter positiu. Sense informació
addicional que els desmenteixi, els conservem i assenyalem
aquesta limitació, en lloc d’aplicar un mínim arbitrari.</p>

</div>

### 6.5. Nombre de pàgines per gènere

Calculem el nombre de registres, el mínim, el màxim, la mitjana
i la mediana de pàgines de cada gènere.

La mitjana i la mediana resumeixen els valors de cada grup,
mentre que el mínim i el màxim mostren els extrems observats.
Cada registre té el mateix pes, sense ponderar pels exemplars.

In [21]:
pagines_per_genere = {}

# Recollim els valors de pàgines de cada gènere.
for registre in registres_nets:
    genere = registre["genere"]

    if genere not in pagines_per_genere:
        pagines_per_genere[genere] = []

    pagines_per_genere[genere].append(registre["pagines"])

# Preparem les estadístiques de cada grup.
resum_pagines_genere = {}

for genere, valors in pagines_per_genere.items():
    resum_pagines_genere[genere] = {
        "registres": len(valors),
        "minim": min(valors),
        "maxim": max(valors),
        "mitjana": mean(valors),
        "mediana": median(valors)
    }

# Ordenem de més a menys mitjana de pàgines.
pagines_generes_ordenats = sorted(
    resum_pagines_genere.items(),
    key=lambda element: element[1]["mitjana"],
    reverse=True
)

print("PÀGINES PER GÈNERE\n")
print(
    f"{'Gènere':<18} {'Registres':>10} {'Mínim':>8} "
    f"{'Màxim':>8} {'Mitjana':>10} {'Mediana':>10}"
)

for genere, dades in pagines_generes_ordenats:
    print(
        f"{genere:<18} {dades['registres']:>10} "
        f"{dades['minim']:>8} {dades['maxim']:>8} "
        f"{dades['mitjana']:>10.2f} {dades['mediana']:>10.2f}"
    )

# Comprovem que cada grup tingui el nombre de registres esperat.
for genere, dades in resum_pagines_genere.items():
    assert dades["registres"] == resum_genere[genere]["registres"], (
        f"El nombre de registres de {genere} no coincideix."
    )

# La suma de pàgines s'utilitza aquí només com a control del reagrupament.
assert sum(
    sum(valors) for valors in pagines_per_genere.values()
) == sum(pagines), "El reagrupament ha alterat els valors de pàgines."

print("\nLes agrupacions coincideixen amb les dades analitzades.")

PÀGINES PER GÈNERE

Gènere              Registres    Mínim    Màxim    Mitjana    Mediana
Ficció                     12       12       80      41.58      40.00
Coneixement                24        2       79      37.83      40.50
Educació                   15        6       75      37.73      38.00
Còmic                      14        1       74      36.71      37.50
Infantil                   13        4       80      34.92      32.00
Papereria                  12        1       65      33.33      31.50

Les agrupacions coincideixen amb les dades analitzades.


<div style="color: #5DADE2; font-style: italic;">

<p>Ficció presenta la mitjana més alta, amb 41,58 pàgines per
registre, mentre que Papereria té la més baixa, amb 33,33.
La diferència entre aquestes mitjanes és de 8,25 pàgines.</p>

<p>Coneixement i Educació mostren mitjanes molt properes,
de 37,83 i 37,73 pàgines. Còmic se situa en 36,71
i Infantil en 34,92 pàgines per registre.</p>

<p>L’ordre canvia lleugerament si observem les medianes:
Coneixement té la més alta, amb 40,50 pàgines, seguit de
Ficció, amb 40. Això mostra que ordenar per mitjana o
per mediana pot donar resultats diferents.</p>

<p>Els intervals entre mínim i màxim se solapen àmpliament.
Per exemple, Ficció presenta entre 12 i 80 pàgines i
Infantil entre 4 i 80. Per tant, una mitjana superior
no significa que tots els registres d’un gènere tinguin
més pàgines que els d’un altre.</p>

<p>Els grups contenen entre 12 i 24 registres. Les estadístiques
descriuen aquests registres acceptats, sense ponderar pels
exemplars, i no permeten generalitzar les diferències a tots
els llibres de cada gènere.</p>

<p>Les comprovacions confirmen que els recomptes per gènere
coincideixen amb els anteriors i que el reagrupament conserva
la suma dels valors de pàgines.</p>

</div>

## 7. Resultats i conclusions

### 7.1. Quin problema hem abordat?

El fitxer de la llibreria barrejava registres correctes amb camps
buits, nombres incorrectes i dates invàlides. Abans de comparar
els exemplars o estudiar les pàgines, calia determinar quines
dades podíem utilitzar.

El treball tenia dos objectius: identificar els problemes de
qualitat i obtenir informació dels registres que complissin
les regles de validació.

### 7.2. Què hem fet?

Hem explorat els 120 registres i definit criteris per als camps
obligatoris, els nombres enters positius i les dates.

Hem normalitzat els formats recuperables, separat els registres
acceptats dels rebutjats i documentat les incidències. Hem conservat
les dades originals, sense inventar informació ni reconstruir
valors per intuïció.

Després hem comparat els exemplars per gènere i sucursal, creuat
aquestes dues classificacions i estudiat les pàgines dels registres
acceptats. Les comprovacions automàtiques han verificat la
classificació de les files i la coherència dels totals.

Finalment, hem exportat els registres acceptats, les incidències
i els registres originals rebutjats a tres fitxers CSV. Els hem
tornat a llegir per comprovar que les columnes i tots els valors
coincideixin amb les dades preparades per exportar.

### 7.3. Què hem descobert sobre la qualitat de les dades?

S’han acceptat **90 registres i rebutjat 30**. Per tant, el **25 %**
de les files incompleix almenys una regla.

S’han documentat 49 motius de rebuig i 9 correccions de format.
Els camps amb més motius de rebuig són `pagines` i `data`,
amb 12 cadascun, seguits d’`exemplars`, amb 11.

Aquest resultat mostra la necessitat de validar millor la informació
quan s’introdueix. Les correccions de format resolen alguns casos,
però no permeten recuperar dades absents o ambigües.

### 7.4. Què hem descobert sobre els exemplars i les pàgines?

Els registres acceptats acumulen **3.377 exemplars**:

- **Coneixement** concentra més exemplars globalment: 831,
  el 24,61 % del total.
- **Vic** és la sucursal amb més exemplars registrats: 1.032,
  el 30,56 %.
- El gènere predominant canvia segons la sucursal: Coneixement
  destaca a Barcelona, Manresa i Tarragona; Còmic a Girona,
  i Educació a Vic.

El creuament mostra diferències que els totals globals amaguen.
També hem comprovat que tenir més registres no implica acumular
més exemplars: Manresa té més files acceptades que Vic, però
menys exemplars.

Les pàgines van d’1 a 80, amb una mitjana de 37,12 i una mediana
de 38. Ficció presenta la mitjana més alta i Papereria la més baixa,
però els rangs dels gèneres se solapen àmpliament. Aquests càlculs
descriuen les pàgines per registre, sense ponderar pels exemplars
ni considerar cada fila un llibre únic.

### 7.5. Per a què serveix aquesta anàlisi?

El treball proporciona un **diagnòstic de qualitat** i una
**descripció dels registres acceptats**.

Permet identificar quins camps necessiten controls d’entrada
i comparar com es distribueixen els exemplars entre gèneres
i sucursals. També ofereix un procediment reproduïble per repetir
l’anàlisi quan es disposi de dades corregides.

Els CSV generats permeten consultar les dades netes i revisar
tots els registres rebutjats amb els seus motius. El camp
`fila_csv` manté la relació amb el fitxer original i facilita
la revisió i una possible correcció en origen.

### 7.6. Quins límits tenen les conclusions?

Els resultats corresponen als 90 registres acceptats. Els 30 rebutjats
podrien modificar les comparacions, i no hem estudiat si afecten
de manera desigual els gèneres o les sucursals.

El fitxer tampoc defineix si els exemplars corresponen a vendes,
entrades d’estoc o altres operacions. Per això, els resultats
no permeten decidir quins llibres comprar, deduir preferències
dels clients ni valorar el rendiment de les sucursals.

Finalment, superar les regles de validació no garanteix que una dada
sigui certa en origen. Els valors poc habituals, com una sola pàgina,
necessitarien una font de contrast per confirmar-los o corregir-los.

## 8. Possibles millores

### 8.1. Millores en la recollida de dades

- **Impedir camps obligatoris buits:** comprovar que `titol`,
  `genere` i `sucursal` continguin informació després d’eliminar
  els espais dels extrems.

- **Validar les quantitats abans de desar-les:** exigir enters
  estrictament positius a `exemplars` i `pagines`, amb un missatge
  que expliqui l’error i permeti corregir-lo en origen.

- **Utilitzar un selector de dates:** evitar dates impossibles
  i desar-les sempre amb el mateix format.

- **Normalitzar els formats d’entrada:** eliminar espais sobrants
  i establir un criteri únic per representar els nombres.

### 8.2. Millores en la definició de les dades

- **Documentar què representa cada registre:** aclarir el significat
  de la quantitat d’exemplars i incorporar el tipus d’operació
  si el fitxer recull operacions diferents.

- **Identificar els llibres:** incorporar un identificador únic
  i, quan correspongui, l’ISBN i l’edició. Les denominacions actuals,
  com «Novel·la» o «Quadern», no permeten distingir obres concretes
  ni comprovar les seves característiques.

- **Definir les classificacions:** documentar els criteris de gènere
  i les combinacions permeses, per poder detectar inconsistències
  sense corregir-les per intuïció.

### 8.3. Millores en una futura anàlisi

- **Estudiar la distribució dels rebutjos:** comprovar si es concentren
  en determinats gèneres o sucursals, mantenint un grup separat
  quan la classificació sigui buida.

- **Contrastar els valors poc habituals amb la font original:**
  revisar casos com els registres d’una pàgina sense eliminar-los
  automàticament ni inventar límits.

- **Recuperar informació corregida en origen:** tornar a executar
  l’anàlisi quan es disposi dels registres esmenats i comprovar
  si canvien els resultats.

- **Afegir context segons la decisió que es vulgui prendre:**
  per estudiar reposicions, per exemple, caldrien dades identificades
  de vendes i existències, a més dels exemplars registrats.